# Step 5: Profile to moisturizer recommendations

## 1. Import and check the baseline

Load the reusable baseline module and verify that it returns
the expected recommendations from the exported product data.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Make the project accessible from the notebooks folder.
project_folder = Path.cwd().parent
sys.path.insert(0, str(project_folder))

from models.baseline import rank_moisturizers, ranking_data

result = rank_moisturizers(ranking_data)

print(result['status'])
pd.DataFrame(result['results'])

ok


,code,product_name,moisturising_flag,has_acne_active,score,reason
0,3401360106994,Sébium sensitive,True,True,2,MOISTURISING ingredients detected: mannitol. P...
1,26023717,Anti-âge Soin de jour Q10 Active,False,True,1,Positive acne-related ingredient flag in the c...
2,3178040554931,Expert Rides 3D Activateur Hyaluronique Crème ...,True,False,1,MOISTURISING ingredients detected: dimethylsil...


## 2. Apply ingredient-avoidance preferences

Exclude products flagged as containing parfum or essential oils
when the corresponding preference is selected.

For each requested exclusion, retain only products explicitly
flagged False. Unknown values are not treated as absence.

In [3]:
def filter_avoid(data, avoid):
    selected = data.copy()
    
    if 'fragrance' in avoid:
        selected = selected[
            selected['has_parfum'] == False
        ]
        
    if 'essential_oils' in avoid:
        selected = selected[
            selected['has_essential_oil'] == False
        ]
        
    return selected

In [4]:
selected_products = filter_avoid(
    ranking_data,
    avoid=['fragrance']
)

print('Before filtering:', len(ranking_data))
print('After filtering:', len(selected_products))

result = rank_moisturizers(selected_products)

print(result['status'])
pd.DataFrame(result['results'])

Before filtering: 50
After filtering: 8
ok


,code,product_name,moisturising_flag,has_acne_active,score,reason
0,3600550880687,Crème riche apaisante pro-tolérance,False,True,1,Positive acne-related ingredient flag in the c...
1,3605970360757,Ultra Facial Cream,False,True,1,Positive acne-related ingredient flag in the c...
2,3701129804445,Créaline Défensive,True,False,1,MOISTURISING ingredients detected: mannitol


### Fragrance-avoidance result

Selecting fragrance avoidance retains 8 of the 50 products.
The baseline returns three alternatives from this filtered pool.

## 3. Check avoidance combinations

Test no avoidance, fragrance avoidance, essential-oil avoidance
and both preferences. Verify that returned products respect
the requested exclusions.

In [7]:
avoid_options = [
    [],
    ['fragrance'],
    ['essential_oils'],
    ['fragrance', 'essential_oils']
]

for avoid in avoid_options:
    selected = filter_avoid(ranking_data, avoid)
    
    if 'fragrance' in avoid:
        assert selected['has_parfum'].eq(False).all()
    
    if 'essential_oils' in avoid:
        assert selected['has_essential_oil'].eq(False).all()
        
    
    result = rank_moisturizers(selected)
    
    print(
        'Avoid:', avoid,
        ' - Eligible:', len(selected),
        ' - Returned:', len(result['results']),
        ' - Status:', result['status']
    )

Avoid: []  - Eligible: 50  - Returned: 3  - Status: ok
Avoid: ['fragrance']  - Eligible: 8  - Returned: 3  - Status: ok
Avoid: ['essential_oils']  - Eligible: 46  - Returned: 3  - Status: ok
Avoid: ['fragrance', 'essential_oils']  - Eligible: 7  - Returned: 3  - Status: ok


### Avoidance checks passed

| Avoidance preference | Eligible products |
|---|---:|
| None | 50 |
| Fragrance | 8 |
| Essential oils | 46 |
| Both | 7 |

All checks passed. Each selection returns three alternatives.

## 4. Check questionnaire scope

Check questions 1 to 4 in order.
Return referral or out_of_scope when required.
Later answers are not needed after an early stop.

In [12]:
def check_scope(profile):
    if type(profile.get('is_adult')) is not bool:
        return 'incomplete_profile'
    
    if profile['is_adult'] == False:
        return 'referral'
    
    if profile.get('concern') not in ['acne_prone', 'other']:
        return 'incomplete_profile'
    
    if profile['concern'] == 'other':
        return 'out_of_scope'
    
    level = profile.get('blemish_level')
    
    if type(level) is not int or level not in [1, 2, 3, 4, 5]:
        return 'incomplete profile'
    
    if level >= 4:
        return 'referral'
    
    flags = profile.get('referral_flags')
    
    if not isinstance(flags, list):
        return 'incomplete_profile'
    
    valid_flags = [
        'prescribed_treatment',
        'unusual_lesion',
        'pregnant_or_breastfeeding'
    ]
    
    for flag in flags:
        if flag not in valid_flags:
            return 'incomplete_profile'
        
    if flags:
        return 'referral'
    
    return 'eligible'

In [13]:
test_profiles = [
    {"is_adult": False},
    {"is_adult": True, "concern": "other"},
    {
        "is_adult": True,
        "concern": "acne_prone",
        "blemish_level": 4
    },
    {
        "is_adult": True,
        "concern": "acne_prone",
        "blemish_level": 3,
        "referral_flags": []
    }
]

for profile in test_profiles:
    print(check_scope(profile))

referral
out_of_scope
referral
eligible


## 5. Connect the profile to baseline recommendations

Check scope before requesting later answers.
For eligible profiles, validate skin type, reactivity and
avoidance preferences, then filter and rank products.

Skin type and reactivity are validated but do not yet
affect the current baseline score.

In [34]:
def recommend(profile, data):
    status = check_scope(profile)
    
    if status != 'eligible':
        return {
            'status': status,
            'message': outcome_message(profile, status),
            'results': []
        }
    
    valid_skin_types = [
        'oily', 'combination', 'normal', 'dry', 'not_sure'    
    ]
    
    if profile.get('skin_type') not in valid_skin_types:
        return {
            'status': 'incomplete_profile',
            "message": "Please complete the questionnaire with valid answers.",
            'results': []
        }
        
    if type(profile.get('reactivity')) is not bool:
        return {
            'status': 'incomplete_profile',
            "message": "Please complete the questionnaire with valid answers.",
            'results': []
        }
    
    avoid = profile.get('avoid', [])
    
    if not isinstance(avoid, list):
        return {
            'status': 'incomplete_profile',
            "message": "Please complete the questionnaire with valid answers.",
            'results': []
        }
    
    for preference in avoid:
        if preference not in ['fragrance', 'essential_oils']:
            return {
                'status': 'incomplete_profile',
                "message": "Please complete the questionnaire with valid answers.",
                'results': []
            }
            
    selected = filter_avoid(data, avoid)
    
    return rank_moisturizers(selected)

In [35]:
profile = {
    "is_adult": True,
    "concern": "acne_prone",
    "blemish_level": 2,
    "referral_flags": [],
    "skin_type": "oily",
    "reactivity": True,
    "avoid": ["fragrance"]
}

result = recommend(profile, ranking_data)

print(result["status"])
pd.DataFrame(result["results"])

ok


,code,product_name,moisturising_flag,has_acne_active,score,reason
0,3600550880687,Crème riche apaisante pro-tolérance,False,True,1,Positive acne-related ingredient flag in the c...
1,3605970360757,Ultra Facial Cream,False,True,1,Positive acne-related ingredient flag in the c...
2,3701129804445,Créaline Défensive,True,False,1,MOISTURISING ingredients detected: mannitol


## 6. Add questionnaire outcome messages

Provide the agreed messages for incomplete, out-of-scope
and referral outcomes.

When several Q4 referral flags apply, include each
corresponding message in questionnaire order.

In [36]:
def outcome_message(profile, status):
    if status == "incomplete_profile":
        return "Please complete the questionnaire with valid answers."

    if status == "out_of_scope":
        return "Swan only covers acne-prone skin for now."

    if profile["is_adult"] == False:
        return (
            "Swan is made for adults. "
            "Please ask a specialized doctor if needed."
        )

    if profile["blemish_level"] >= 4:
        return (
            "Lots of red, painful or deep spots, or scars, "
            "are worth showing to a doctor or dermatologist."
        )

    referral_messages = {
        "prescribed_treatment": (
            "Check with the person who prescribed your treatment "
            "before adding new products."
        ),
        "unusual_lesion": (
            "A sudden or unusual change in your skin "
            "should be checked by a doctor."
        ),
        "pregnant_or_breastfeeding": (
            "This MVP does not suggest products during pregnancy "
            "or breastfeeding. Please ask a pharmacist or your "
            "healthcare professional for advice."
        )
    }

    messages = []

    for flag, message in referral_messages.items():
        if flag in profile["referral_flags"]:
            messages.append(message)

    return " ".join(messages)

In [37]:
result = recommend({"is_adult": False}, ranking_data)

print(result["status"])
print(result["message"])

referral
Swan is made for adults. Please ask a specialized doctor if needed.


## 7. Test questionnaire routing and messages

Check all referral conditions, out-of-scope routing,
incomplete profiles and multiple Q4 flags.

In [38]:
base_profile = {
    "is_adult": True,
    "concern": "acne_prone",
    "blemish_level": 2,
    "referral_flags": [],
    "skin_type": "oily",
    "reactivity": False,
    "avoid": []
}

test_cases = [
    ("Under 18", {"is_adult": False}, "referral"),
    ("Other concern", {"concern": "other"}, "out_of_scope"),
    ("Level 4", {"blemish_level": 4}, "referral"),
    ("Level 5", {"blemish_level": 5}, "referral"),
    (
        "Prescribed treatment",
        {"referral_flags": ["prescribed_treatment"]},
        "referral"
    ),
    (
        "Worrying change",
        {"referral_flags": ["unusual_lesion"]},
        "referral"
    ),
    (
        "Pregnancy or breastfeeding",
        {"referral_flags": ["pregnant_or_breastfeeding"]},
        "referral"
    ),
    (
        "Two referral flags",
        {
            "referral_flags": [
                "prescribed_treatment",
                "pregnant_or_breastfeeding"
            ]
        },
        "referral"
    ),
    ("Missing age", {"is_adult": None}, "incomplete_profile"),
    ("Level 3", {"blemish_level": 3}, "ok")
]

for name, changes, expected in test_cases:
    profile = base_profile.copy()
    profile.update(changes)

    result = recommend(profile, ranking_data)

    assert result["status"] == expected, name

    if expected != "ok":
        assert result["results"] == [], name
        assert result.get("message"), name

    print(name, "-", result["status"])

Under 18 - referral
Other concern - out_of_scope
Level 4 - referral
Level 5 - referral
Prescribed treatment - referral
Worrying change - referral
Pregnancy or breastfeeding - referral
Two referral flags - referral
Missing age - incomplete_profile
Level 3 - ok


## 8. Check optional answers and insufficient results

Verify that omitting Q7 is equivalent to selecting no avoidance.
Check that the full recommendation flow returns available
products when fewer than three remain, and handles an empty pool.

In [39]:
# Q7 is optional.
profile_without_avoid = base_profile.copy()
profile_without_avoid.pop("avoid")

assert (
    recommend(profile_without_avoid, ranking_data)
    == recommend(base_profile, ranking_data)
)

print("Optional Q7: passed")


# Test the full flow with smaller product pools.
for number in [0, 1, 2]:
    small_catalogue = ranking_data.head(number)

    result = recommend(base_profile, small_catalogue)

    expected_status = (
        "no_result" if number == 0 else "insufficient_results"
    )

    assert result["status"] == expected_status
    assert len(result["results"]) == number

    print(
        "Available:", number,
        "| Status:", result["status"],
        "| Returned:", len(result["results"])
    )

Optional Q7: passed
Available: 0 | Status: no_result | Returned: 0
Available: 1 | Status: insufficient_results | Returned: 1
Available: 2 | Status: insufficient_results | Returned: 2


In [40]:
for field, invalid_value in [
    ("skin_type", None),
    ("reactivity", None),
    ("avoid", "fragrance"),
    ("avoid", ["unknown"])
]:
    profile = base_profile.copy()
    profile[field] = invalid_value

    result = recommend(profile, ranking_data)

    assert result["status"] == "incomplete_profile"
    assert result["message"]
    assert result["results"] == []

    print(field, "-", invalid_value, "- passed")

skin_type - None - passed
reactivity - None - passed
avoid - fragrance - passed
avoid - ['unknown'] - passed


## 9. Prepare the recommendation module

Move the tested profile checks, avoidance filters and
recommendation flow into a reusable Python module.

In [42]:
from models.recommendations import recommend as recommend_from_module

module_result = recommend_from_module(base_profile, ranking_data)
notebook_result = recommend(base_profile, ranking_data)

assert module_result == notebook_result

print("Module matches notebook: passed")

Module matches notebook: passed


### Module check result

The imported recommendation function returns the same result
as the notebook function for the tested profile.

## Decisions and points for team review

| Section | Decision made | Status |
|---|---|---|
| 2–3 | Apply fragrance avoidance using has_parfum only and essential-oil avoidance using has_essential_oil. Unknown flag values are not treated as absence. | Implemented and tested |
| 4 | Check Q1–Q4 in questionnaire order. Stop at referral or out-of-scope outcomes without requiring later answers. | Implemented and tested |
| 5 | Require valid skin_type and reactivity for eligible profiles. These fields do not yet affect ranking. | Validation implemented; personalisation pending |
| 5, 8 | Treat omitted Q7 as an empty avoidance list. Reject invalid avoidance values. | Implemented and tested |
| 6 | Use the questionnaire messages. When multiple Q4 flags apply, include their messages in questionnaire order. | Implemented; combined-message policy to be reviewed |
| 7–8 | Test routing, invalid answers, optional Q7 and empty or insufficient product pools. | Tests passed |
| 9 | Make the recommendation flow available through models/recommendations.py. | Module matches notebook for the tested profile |